# EDA — Marche_Principal_Droits_MERGED.csv

In [1]:
import pandas as pd
import numpy as np

df_droits = pd.read_csv('D:/rouge_file/DATA_BULLETINS_CSV/Marche_Principal_Droits_MERGED.csv')


In [2]:
print(f'Shape : {df_droits.shape}')

Shape : (13678, 12)


In [3]:
df_droits.head(5)

,Type,Nombre de titres,Référence,Unnamed: 3,Instruments,Unnamed: 5,Cours du jour,Statut,Meilleurs à la clôture,Unnamed: 9,Depuis janvier,Unnamed: 11
0,Type,NaN,Cours,Date,Code ISIN,Désignation,NaN,NaN,Dem.,Off.,+Haut,+Bas
1,Principal A / Attributions,1 657 595,92,03/10/22,MA0000800682,DA BCI 1/5 2006,-,NT,-,-,-,-
2,Principal A / Attributions,14 431 940,"16,53",03/10/22,MA0000801185,DA BOA 1/10 2000,-,NT,-,"17,70",-,-
3,Principal A / Attributions,205 606 648,"2,55",03/10/22,MA0000801219,DA BOA 1/65 2022,"2,54",T,-,-,"2,54","2,54"
4,Principal A / Attributions,28 571 420,"47,23",03/10/22,MA0000801177,DA BOA 2/7 1996,-,NT,-,"50,57",-,-


## 1. Qualite des donnees

In [4]:
print('Valeurs manquantes :')
print(df_droits.isnull().sum())
print(f'\nDoublons         : {df_droits.duplicated().sum()}')
print(f'Lignes parasites : {(df_droits["Type"] == "Type").sum()}')
print('\nValeurs colonne Type :')
print(df_droits['Type'].value_counts())

Valeurs manquantes :
Type                        0
Nombre de titres          679
Référence                   0
Unnamed: 3                  9
Instruments                 9
Unnamed: 5                 20
Cours du jour             699
Statut                    699
Meilleurs à la clôture     20
Unnamed: 9                 20
Depuis janvier             20
Unnamed: 11                20
dtype: int64

Doublons         : 6480
Lignes parasites : 682

Valeurs colonne Type :
Type
Principal B / Attributions    9512
Principal A / Attributions    3419
Type                           682
Souscriptions                   48
Non spécifié                    17
Name: count, dtype: int64


## 2. Nettoyage

In [5]:
df_droits = df_droits[df_droits['Type'] != 'Type'].drop_duplicates().copy()

df_droits = df_droits.rename(columns={
    'Unnamed: 3'  : 'Date_Seance',
    'Unnamed: 5'  : 'Designation',
    'Unnamed: 9'  : 'Meilleur_Offre',
    'Unnamed: 11' : 'Plus_Bas_Janv',
    'Depuis janvier'          : 'Plus_Haut_Janv',
    'Meilleurs a la cloture'  : 'Meilleur_Demande',
})

def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace(' ','').str.replace('\xa0','')
         .str.replace(',','.').str.replace('-','').str.strip(),
        errors='coerce'
    )

for col in ['Nombre de titres','Reference','Cours du jour','Plus_Haut_Janv','Plus_Bas_Janv','Meilleur_Offre']:
    if col in df_droits.columns:
        df_droits[col] = to_num(df_droits[col])

if 'Reference' not in df_droits.columns and 'Référence' in df_droits.columns:
    df_droits['Reference'] = to_num(df_droits['Référence'])

df_droits['Date_Seance'] = pd.to_datetime(df_droits['Date_Seance'], format='%d/%m/%y', errors='coerce')

print(f'Shape apres nettoyage : {df_droits.shape}')
df_droits.head(5)

Shape apres nettoyage : (7193, 13)


,Type,Nombre de titres,Référence,Date_Seance,Instruments,Designation,Cours du jour,Statut,Meilleurs à la clôture,Meilleur_Offre,Plus_Haut_Janv,Plus_Bas_Janv,Reference
1,Principal A / Attributions,1657595.0,92,2022-10-03,MA0000800682,DA BCI 1/5 2006,NaN,NT,-,NaN,NaN,NaN,92.00
2,Principal A / Attributions,14431940.0,"16,53",2022-10-03,MA0000801185,DA BOA 1/10 2000,NaN,NT,-,17.70,NaN,NaN,16.53
3,Principal A / Attributions,205606648.0,"2,55",2022-10-03,MA0000801219,DA BOA 1/65 2022,2.54,T,-,NaN,2.54,2.54,2.55
4,Principal A / Attributions,28571420.0,"47,23",2022-10-03,MA0000801177,DA BOA 2/7 1996,NaN,NT,-,50.57,NaN,NaN,47.23
5,Principal A / Attributions,4764303.0,4 293,2022-10-03,MA0000800781,DA LHM 8/3 2007,NaN,NT,-,NaN,NaN,NaN,4293.00


## 3. Statistiques descriptives

In [6]:
df_droits[['Nombre de titres','Référence','Cours du jour','Plus_Haut_Janv','Plus_Bas_Janv']].describe().round(2)

,Nombre de titres,Cours du jour,Plus_Haut_Janv,Plus_Bas_Janv
count,7.176000e+03,409.00,1445.00,1445.00
mean,3.798080e+07,20.71,180.08,177.85
std,6.620357e+07,260.97,759.10,759.05
min,3.208300e+04,0.39,0.39,0.39
25%,1.100000e+06,2.93,3.53,3.05
50%,4.764303e+06,3.43,21.10,20.94
75%,2.857142e+07,4.09,113.00,113.00
max,2.157863e+08,5257.00,5257.00,5257.00


## 4. Analyse par Type de Droit

In [8]:
type_count = df_droits['Type'].value_counts()
type_pct   = (type_count / type_count.sum() * 100).round(2)
print(pd.DataFrame({'Nb': type_count, '%': type_pct}).to_string())
print()
print('Nombre de droits distincts par type :')
print(df_droits.groupby('Type')['Designation'].nunique().to_string())

                              Nb      %
Type                                   
Principal B / Attributions  4268  59.34
Principal A / Attributions  2860  39.76
Souscriptions                 48   0.67
Non spécifié                  17   0.24

Nombre de droits distincts par type :
Type
Non spécifié                   0
Principal A / Attributions     9
Principal B / Attributions    15
Souscriptions                  3


## 5. Analyse du Statut

In [9]:
statut_count = df_droits['Statut'].value_counts()
statut_pct   = (statut_count / statut_count.sum() * 100).round(2)
print(pd.DataFrame({'Nb': statut_count, '%': statut_pct}).to_string())
print('\nLegende : T=Traite | NT=Non Traite | S=Suspendu')
print()
print('Statut par type de droit :')
print(df_droits.groupby(['Type','Statut']).size().unstack(fill_value=0).to_string())

          Nb      %
Statut             
NT      6766  94.29
T        409   5.70
S          1   0.01

Legende : T=Traite | NT=Non Traite | S=Suspendu

Statut par type de droit :
Statut                        NT  S    T
Type                                    
Principal A / Attributions  2504  0  356
Principal B / Attributions  4259  1    8
Souscriptions                  3  0   45


## 6. Analyse des Cours

In [10]:
traites = df_droits[df_droits['Statut'] == 'T']
print(f'Droits traites : {len(traites):,}')
print()
print('Top 10 cours du jour les plus eleves :')
print(traites.nlargest(10,'Cours du jour')[['Designation','Type','Cours du jour','Référence']].to_string(index=False))
print()
print('Top 10 nombre de titres :')
print(df_droits.nlargest(10,'Nombre de titres')[['Designation','Type','Nombre de titres']].to_string(index=False))

Droits traites : 409

Top 10 cours du jour les plus eleves :
    Designation                       Type  Cours du jour Référence
DA LHM 8/3 2007 Principal A / Attributions        5257.00     5 067
DA SOT 1/5 2005 Principal B / Attributions         360.00       360
DA SOT 1/5 2005 Principal B / Attributions         340.30       364
DA BCI 1/5 2006 Principal A / Attributions         113.00     113,4
DA MLE 1/4 1999 Principal B / Attributions          94.99     94,88
DA MLE 1/4 1999 Principal B / Attributions          94.95        90
DA MLE 1/4 1999 Principal B / Attributions          90.90      96,7
DA MLE 1/4 1999 Principal B / Attributions          87.75        89
DA BOA 2/7 1996 Principal A / Attributions          72.00     72,82
DA BOA 2/7 1996 Principal A / Attributions          69.71     67,43

Top 10 nombre de titres :
     Designation                       Type  Nombre de titres
DA BOA 1/48 2025 Principal A / Attributions       215786333.0
DA BOA 1/48 2025 Principal A / Attributi

## 7. Outliers

In [11]:
cols_check = []
for col in ['Nombre de titres', 'Référence', 'Cours du jour']:
    if df_droits[col].dtype in ['float64', 'int64']:
        Q1, Q3 = df_droits[col].quantile(0.25), df_droits[col].quantile(0.75)
        IQR    = Q3 - Q1
        out    = df_droits[(df_droits[col] < Q1-1.5*IQR) | (df_droits[col] > Q3+1.5*IQR)]
        print(f'{col:25s} : {len(out):,} outliers ({len(out)/len(df_droits)*100:.1f}%)')
    else:
        print(f'{col:25s} : non numerique — conversion requise')

Nombre de titres          : 1,277 outliers (17.8%)
Référence                 : non numerique — conversion requise
Cours du jour             : 54 outliers (0.8%)


## 8. Correlation

In [12]:
cols_num = [col for col in ['Nombre de titres','Référence','Cours du jour',
                             'Plus_Haut_Janv','Plus_Bas_Janv']
            if col in df_droits.columns and df_droits[col].dtype in ['float64','int64']]

print(f'Colonnes numeriques disponibles : {cols_num}')
df_droits[cols_num].corr().round(4)

Colonnes numeriques disponibles : ['Nombre de titres', 'Cours du jour', 'Plus_Haut_Janv', 'Plus_Bas_Janv']


,Nombre de titres,Cours du jour,Plus_Haut_Janv,Plus_Bas_Janv
Nombre de titres,1.0000,-0.1635,-0.2138,-0.2119
Cours du jour,-0.1635,1.0000,1.0000,1.0000
Plus_Haut_Janv,-0.2138,1.0000,1.0000,1.0000
Plus_Bas_Janv,-0.2119,1.0000,1.0000,1.0000


## 9. Export

In [13]:


print('Colonnes avant renommage:')
print(list(df_droits.columns))

df_droits = df_droits.rename(columns={
    'Unnamed: 3'             : 'Date_Seance',
    'Unnamed: 5'             : 'Designation',
    'Unnamed: 9'             : 'Meilleur_Offre',
    'Unnamed: 11'            : 'Plus_Bas_Annee',
    'Depuis janvier'         : 'Plus_Haut_Annee',
    'Meilleurs à la clôture' : 'Meilleur_Demande',
})

print()
print('Colonnes apres renommage:')
print(list(df_droits.columns))

df_droits.to_csv('Droits_RENAMED.csv', index=False, encoding='utf-8')
print()
print(f'Exporte : Droits_RENAMED.csv — {len(df_droits):,} lignes')

Colonnes avant renommage:
['Type', 'Nombre de titres', 'Référence', 'Date_Seance', 'Instruments', 'Designation', 'Cours du jour', 'Statut', 'Meilleurs à la clôture', 'Meilleur_Offre', 'Plus_Haut_Janv', 'Plus_Bas_Janv', 'Reference']

Colonnes apres renommage:
['Type', 'Nombre de titres', 'Référence', 'Date_Seance', 'Instruments', 'Designation', 'Cours du jour', 'Statut', 'Meilleur_Demande', 'Meilleur_Offre', 'Plus_Haut_Janv', 'Plus_Bas_Janv', 'Reference']

Exporte : Droits_RENAMED.csv — 7,193 lignes
